# Multi-Panel Visualization for Neural Navigators Project

This notebook creates multi-panel layouts for the key figures from the Neural Navigators project.

In [ ]:
# Basic imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore, ttest_ind
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
import shap

# Set consistent font sizes for plots
plt.rcParams['font.size'] = 10  # Default font size for axis labels
plt.rcParams['axes.titlesize'] = 12  # Font size for titles
plt.rcParams['axes.titleweight'] = 'bold'  # Bold titles
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['figure.autolayout'] = True

## Data Loading

First, we need to load the processed data from the original notebook.

In [ ]:
# Load data from the main notebook
# This would be replaced with actual data loading from Final_code.ipynb or saved data files
# For demonstration purposes, we'll use placeholder data

# Example placeholder data
def create_placeholder_data():
    # Create placeholder data for demonstrations
    np.random.seed(42)
    
    # 1. Connectivity data
    n_regions = 10
    conn_matrix = np.random.randn(n_regions, n_regions) * 0.3
    np.fill_diagonal(conn_matrix, 1.0)  # Set diagonal to 1
    conn_matrix = (conn_matrix + conn_matrix.T) / 2  # Make symmetric
    
    regions = ['MOs', 'ACA', 'PL', 'CP', 'ACB', 'VISp', 'VISl', 'CA1', 'CA3', 'Other']
    
    # 2. Behavioral data
    n_trials = 1000
    age_groups = np.random.choice(['young', 'mature', 'late'], size=n_trials)
    contrasts = np.random.choice([0.1, 0.25, 0.5, 1.0], size=n_trials)
    
    # Make reaction times dependent on age and contrast
    rt_base = {
        'young': 0.25,
        'mature': 0.3,
        'late': 0.4
    }
    
    rt = np.array([rt_base[age] - 0.1 * contrast + 0.05 * np.random.randn() 
                   for age, contrast in zip(age_groups, contrasts)])
    
    # Make accuracy dependent on age and contrast
    acc_base = {
        'young': 0.85,
        'mature': 0.8,
        'late': 0.7
    }
    
    accuracy = np.array([acc_base[age] + 0.15 * contrast - 0.1 * np.random.random() 
                         for age, contrast in zip(age_groups, contrasts)])
    accuracy = np.clip(accuracy, 0, 1)  # Clip to valid range
    
    # 3. Model data
    # Model accuracy
    model_acc = {
        'young': 0.78,
        'mature': 0.72,
        'late': 0.65
    }
    
    # Feature importance
    feature_names = ['MOs', 'PFC', 'BG', 'VIS', 'HPC', 'Other']
    n_features = len(feature_names)
    
    feature_imp_young = np.array([0.25, 0.18, 0.22, 0.15, 0.12, 0.08])
    feature_imp_old = np.array([0.15, 0.22, 0.18, 0.20, 0.14, 0.11])
    
    return {
        'connectivity': {
            'matrix': conn_matrix,
            'regions': regions
        },
        'behavior': {
            'age_groups': age_groups,
            'contrasts': contrasts,
            'reaction_time': rt,
            'accuracy': accuracy
        },
        'model': {
            'accuracy': model_acc,
            'feature_names': feature_names,
            'feature_imp_young': feature_imp_young,
            'feature_imp_old': feature_imp_old
        }
    }

# Create placeholder data
data = create_placeholder_data()

## Multi-Panel Figures

Now let's create the multi-panel figures as requested.

### Figure 1: Connectivity Heat-Map + Bar Summary

In [ ]:
def plot_figure1(data):
    """Create Figure 1: Connectivity heat-map + bar summary"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # A: Connectivity Heat-Map
    conn_matrix = data['connectivity']['matrix']
    regions = data['connectivity']['regions']
    
    mask = np.eye(conn_matrix.shape[0], dtype=bool)  # Mask diagonal elements
    sns.heatmap(conn_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1,
                xticklabels=regions, yticklabels=regions, mask=mask, ax=ax1,
                annot_kws={"size": 9})
    ax1.set_title('A: Functional Connectivity Between Brain Regions')
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=90)
    
    # B: Bar Summary
    age_groups = ['young', 'mature', 'late']
    mean_connectivity = [0.45, 0.35, 0.25]  # Placeholder values
    errors = [0.05, 0.04, 0.03]  # Placeholder standard errors
    
    x_pos = np.arange(len(age_groups))
    bars = ax2.bar(x_pos, mean_connectivity, yerr=errors, align='center',
                   color='skyblue', ecolor='black', capsize=10)
    
    # Add value labels
    for i, bar in enumerate(bars):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + errors[i] + 0.02,
                f'{mean_connectivity[i]:.2f}', ha='center', va='bottom')
    
    ax2.set_xlabel('Age Group')
    ax2.set_ylabel('Average Connectivity')
    ax2.set_title('B: Average Connectivity by Age Group')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(['Young', 'Mature', 'Late'])
    ax2.set_ylim(0, max(mean_connectivity) * 1.3)
    
    # Add figure title
    fig.suptitle('Figure 1: Functional Connectivity Analysis', fontsize=14, y=0.98)
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    
    return fig

# Create Figure 1
fig1 = plot_figure1(data)
plt.show()

### Figure 2: Behavioral RT & Accuracy Plots

In [ ]:
def plot_figure2(data):
    """Create Figure 2: Behavioral RT & accuracy plots"""
    # Create DataFrame from the behavioral data
    behavior_df = pd.DataFrame({
        'age_group': data['behavior']['age_groups'],
        'contrast': data['behavior']['contrasts'],
        'reaction_time': data['behavior']['reaction_time'],
        'accuracy': data['behavior']['accuracy']
    })
    
    # Create contrast categories
    contrast_bins = [0, 0.2, 0.4, 0.8, 1.1]
    contrast_labels = ['Very Low', 'Low', 'Medium', 'High']
    behavior_df['contrast_category'] = pd.cut(behavior_df['contrast'], 
                                             bins=contrast_bins, 
                                             labels=contrast_labels)
    
    # Create figure with two panels
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # A: Reaction Time by Age Group and Contrast
    sns.barplot(x='age_group', y='reaction_time', hue='contrast_category', 
                data=behavior_df, ax=ax1, palette='viridis',
                order=['young', 'mature', 'late'],
                hue_order=contrast_labels,
                errorbar=('se'))
    
    ax1.set_xlabel('Age Group')
    ax1.set_ylabel('Reaction Time (s)')
    ax1.set_title('A: Reaction Time by Age Group and Contrast')
    ax1.set_xticklabels(['Young', 'Mature', 'Late'])
    ax1.legend(title='Contrast')
    
    # B: Accuracy by Age Group and Contrast
    sns.barplot(x='age_group', y='accuracy', hue='contrast_category', 
                data=behavior_df, ax=ax2, palette='viridis',
                order=['young', 'mature', 'late'],
                hue_order=contrast_labels,
                errorbar=('se'))
    
    ax2.set_xlabel('Age Group')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('B: Accuracy by Age Group and Contrast')
    ax2.set_xticklabels(['Young', 'Mature', 'Late'])
    ax2.set_ylim(0, 1)
    ax2.legend(title='Contrast')
    
    # Add figure title
    fig.suptitle('Figure 2: Behavioral Performance Analysis', fontsize=14, y=0.98)
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    
    return fig

# Create Figure 2
fig2 = plot_figure2(data)
plt.show()

### Figure 3: Model Accuracy + Age-Split Metrics

In [ ]:
def plot_figure3(data):
    """Create Figure 3: Model accuracy + age-split metrics"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # A: Model Accuracy
    age_groups = ['young', 'mature', 'late']
    accuracies = [data['model']['accuracy'][ag] for ag in age_groups]
    
    x_pos = np.arange(len(age_groups))
    bars = ax1.bar(x_pos, accuracies, align='center', color='coral')
    
    # Add value labels
    for i, bar in enumerate(bars):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{accuracies[i]:.2f}', ha='center', va='bottom')
    
    ax1.set_xlabel('Age Group')
    ax1.set_ylabel('Model Accuracy')
    ax1.set_title('A: LSTM Model Accuracy by Age Group')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(['Young', 'Mature', 'Late'])
    ax1.set_ylim(0, 1)
    ax1.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Chance')
    ax1.legend()
    
    # B: Age-Split Metrics (F1 scores, precision, recall)
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
    # Create placeholder values for demonstration
    young_metrics = [0.78, 0.75, 0.82, 0.78]
    mature_metrics = [0.72, 0.70, 0.75, 0.72]
    late_metrics = [0.65, 0.62, 0.70, 0.65]
    
    x = np.arange(len(metrics))
    width = 0.25
    
    ax2.bar(x - width, young_metrics, width, label='Young', color='#1f77b4')
    ax2.bar(x, mature_metrics, width, label='Mature', color='#ff7f0e')
    ax2.bar(x + width, late_metrics, width, label='Late', color='#2ca02c')
    
    ax2.set_xlabel('Metric')
    ax2.set_ylabel('Score')
    ax2.set_title('B: Classification Metrics by Age Group')
    ax2.set_xticks(x)
    ax2.set_xticklabels(metrics)
    ax2.set_ylim(0, 1)
    ax2.legend()
    
    # Add figure title
    fig.suptitle('Figure 3: Model Performance Analysis', fontsize=14, y=0.98)
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    
    return fig

# Create Figure 3
fig3 = plot_figure3(data)
plt.show()

### Figure 4: Feature-Importance Plots

In [ ]:
def plot_figure4(data):
    """Create Figure 4: Feature-importance plots"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Extract feature importance data
    feature_names = data['model']['feature_names']
    feature_imp_young = data['model']['feature_imp_young']
    feature_imp_old = data['model']['feature_imp_old']
    
    # A: Feature Importance for Young Mice
    # Sort by importance
    sorted_idx = np.argsort(feature_imp_young)
    sorted_features = [feature_names[i] for i in sorted_idx]
    sorted_imp = feature_imp_young[sorted_idx]
    
    # Create horizontal bar plot
    bars = ax1.barh(sorted_features, sorted_imp, color='skyblue')
    
    # Add value labels
    for i, v in enumerate(sorted_imp):
        ax1.text(v + 0.01, i, f'{v:.2f}', va='center')
    
    ax1.set_xlabel('Feature Importance')
    ax1.set_title('A: Feature Importance for Young Mice')
    ax1.set_xlim(0, max(feature_imp_young) * 1.2)
    
    # B: Feature Importance for Old Mice
    # Sort by importance
    sorted_idx = np.argsort(feature_imp_old)
    sorted_features = [feature_names[i] for i in sorted_idx]
    sorted_imp = feature_imp_old[sorted_idx]
    
    # Create horizontal bar plot
    bars = ax2.barh(sorted_features, sorted_imp, color='coral')
    
    # Add value labels
    for i, v in enumerate(sorted_imp):
        ax2.text(v + 0.01, i, f'{v:.2f}', va='center')
    
    ax2.set_xlabel('Feature Importance')
    ax2.set_title('B: Feature Importance for Late Adult Mice')
    ax2.set_xlim(0, max(feature_imp_old) * 1.2)
    
    # Add figure title
    fig.suptitle('Figure 4: Feature Importance Analysis', fontsize=14, y=0.98)
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    
    return fig

# Create Figure 4
fig4 = plot_figure4(data)
plt.show()

## Conclusion

In this notebook, we have created multi-panel visualizations for the Neural Navigators project, following the guidelines to create more compact figures:

1. Figure 1: Connectivity heat-map + bar summary (Fig 1A-B)
2. Figure 2: Behavioral RT & accuracy plots (Fig 2A-B)
3. Figure 3: Model accuracy + age-split metrics (Fig 3A-B)
4. Figure 4: Feature-importance plots (Fig 4A-B)

These visualizations maintain the same information content as the original figures while using a more space-efficient layout.